## Experiment 2 — Pima Indians Diabetes (tabular)

### Dataset
The [Pima Indians Diabetes dataset](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
contains 768 observations of female patients aged 21 or older, with 8 clinical features
and a binary outcome indicating the presence of type 2 diabetes. The dataset is moderately
imbalanced — ~35% positive (diabetes) and ~65% negative.

### Motivation
This experiment tests `XGBOptClf` on raw tabular data where hyperparameter tuning is expected to have a more meaningful impact.

### Results

| Model | Nested CV mean AUC | Nested CV std AUC | Decision ($\tau$) Threshold
|---|---|---|---|
| Default XGBoost | 0.7914 | 0.0284 | - |
| XGBOptClf (n_trials=100, std = 0.5) | 0.8329 | 0.0325 | 0.3674 ± 0.0101
| XGBOptClf (n_trials=100, std = 1)** | 0.8346 | 0.0314 | 0.3627 ± 0.0096 
| Improvement | +0.04 | — | -|


## Conclusions

- Optuna-optimized XGBoost consistently outperforms the default baseline by **+4.3% AUC**
- A higher `std_penalty` (`1.0` vs `0.5`) favours stability over raw performance, yielding a slightly better generalisation (0.8346 vs 0.8329)
- The decision threshold is stable across folds (std < 0.01), indicating a **reliable decision boundary**

## Dev / Test Gap

- `dev_mcc` (0.5984) vs `test_mcc` (0.4710) shows a gap of **0.1274**
- This is expected — the threshold is optimised on dev data via Youden's J, so it naturally fits dev better than test

### Further improvements
The following domain knowledge is known to push the AUC score toward the 0.88–0.94 range for this dataset and is left as future work:

- "Clinically implausible zeros in Glucose, Blood Pressure, Skin Thickness, Insulin, and BMI are treated as missing and imputed with class-conditional medians." [https://theaspd.com/index.php/ijes/article/view/8768/6339]


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import xgboost as xgb
from src.xgb_opt_clf import XGBOptClf
from src.helper_functions import *
import os
import tempfile
import optuna.visualization as vis
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import json

/Users/vkvutov/Documents/my_codes/git_projects/xgb-cat-optuna/xgb-optuna/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Study Setup

In [2]:
N_TRIALS = 100
N_SPLITS = 5
RANDOM_STATE = 42
STUDY_NAME = "xgb_opt_clf_pima"

Throughout this notebook, the development set will refer to the training set and validation sets, while the test set will be used to report results

In [3]:
# Load Pima Indians Diabetes dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"
cols = ["pregnancies", "glucose", "blood_pressure", "skin_thickness", 
        "insulin", "bmi", "diabetes_pedigree", "age", "outcome"]
df = pd.read_csv(url, names=cols)

X = df.drop("outcome", axis=1).values
y = df["outcome"].values

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y)

In [4]:
pd.Series(y).value_counts(normalize=True)

0    0.651042
1    0.348958
Name: proportion, dtype: float64

In [5]:
# XGBOptClf
xgb_opt_clf_default = XGBOptClf(n_trials = N_TRIALS)

xgb_opt_clf_high_penalty = XGBOptClf(n_trials = N_TRIALS, std_penalty=1)

# Nested CV
nested_cv_default = nested_cv_score(xgb_opt_clf_default, X, y, n_outer=5, stratified=True)
print(f"XGBOptClf default nested CV: {nested_cv_default['scores'].mean():.4f} ± {nested_cv_default['scores'].std():.4f}")

nested_cv_high_penalty = nested_cv_score(xgb_opt_clf_high_penalty, X, y, n_outer=5, stratified=True)
print(f"XGBOptClf high penalty nested CV: {nested_cv_high_penalty['scores'].mean():.4f} ± {nested_cv_high_penalty['scores'].std():.4f}")

# Default XGBoost
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
default_clf = xgb.XGBClassifier(random_state=42)
default_scores = cross_val_score(default_clf, X, y, cv=outer_cv, scoring="roc_auc")
print(f"Default XGBoost: {default_scores.mean():.4f} ± {default_scores.std():.4f}")

XGBOptClf default nested CV: 0.8329 ± 0.0325
XGBOptClf high penalty nested CV: 0.8346 ± 0.0314
Default XGBoost: 0.7914 ± 0.0284


In [6]:
print(f"nested_cv_default['optimal_thresholds']: {nested_cv_default['optimal_thresholds']}")
print(f"nested_cv_default['optimal_thresholds'].mean: {np.mean(nested_cv_default['optimal_thresholds']):.4f} ± {np.std(nested_cv_default['optimal_thresholds']):.4f}")
print(f"nested_cv_high_penalty['optimal_thresholds']: {nested_cv_high_penalty['optimal_thresholds']}")
print(f"nested_cv_high_penalty['optimal_thresholds'].mean: {np.mean(nested_cv_high_penalty['optimal_thresholds']):.4f} ± {np.std(nested_cv_high_penalty['optimal_thresholds']):.4f}")

nested_cv_default['optimal_thresholds']: [0.36856997 0.36464649 0.37296444 0.35004166 0.38054517]
nested_cv_default['optimal_thresholds'].mean: 0.3674 ± 0.0101
nested_cv_high_penalty['optimal_thresholds']: [0.35550112 0.35744518 0.38054779 0.35500681 0.36519673]
nested_cv_high_penalty['optimal_thresholds'].mean: 0.3627 ± 0.0096


## Experiment Tracking with MLflow

This cell logs the full experiment to MLflow, including:

- **Parameters** — Optuna hyperparameters
- **Metrics** — CV AUC mean/std, dev/test AUC, dev/test MCC, applied threshold, nested CV scores per fold
- **Artifacts** — classification reports (JSON), ROC curve (PNG), Optuna plots (HTML), trained XGBoost model

### Nested CV
Performance is obtained via nested cross-validation (`n_outer=5`, `std_penalty=1`).
The outer loop evaluates performance; the inner loop (inside `fit()`) optimises hyperparameters via Optuna.

### Viewing Results
Launch the MLflow UI with:
```bash
cd notebooks

mlflow ui --port 5002
```
Then open `http://localhost:5002` in your browser.

In [7]:
import os, json, time, tempfile
import mlflow
import optuna.visualization as vis
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_curve

mlflow.set_tracking_uri(f"file://{os.path.abspath('mlruns')}")
mlflow.set_experiment(STUDY_NAME)

with mlflow.start_run(run_name="pima_diabetes"):

    # ─── Tags 
    mlflow.set_tags({
        "model":   "XGBOptClf",
        "dataset": "pima_diabetes",
    })

    # ─── Fit 
    clf = XGBOptClf(n_trials=N_TRIALS, std_penalty=1)
    clf.fit(X_dev, y_dev)

    # ─── Params 
    mlflow.log_params({
        **clf.best_params_,
        "n_trials":    clf.n_trials,
        "n_estimators": clf.best_num_boost_round_,
        "n_jobs":      clf._resolve_n_jobs(),
        "std_penalty": clf.std_penalty,
        "eval_metric": clf.eval_metric,
        "nfold":       clf.nfold,
    })

    # ─── CV Metrics 
    mlflow.log_metrics({
        "val_auc":     clf.score(X_test, y_test),
        "cv_auc_mean": clf.cv_results_[f"test_{clf.eval_metric}_mean"],
        "cv_auc_std":  clf.cv_results_[f"test_{clf.eval_metric}_std"],
    })

    # ─── Eval Metrics 
    results = clf.eval(X_dev, y_dev, X_test, y_test)
    mlflow.log_metrics({
        "dev_auc":           results["dev_auc"],
        "dev_mcc":           results["dev_mcc"],
        "test_auc":          results["test_auc"],
        "test_mcc":          results["test_mcc"],
        "applied_threshold": results["applied_threshold"],
    })

    # ─── Nested CV 
    nested_cv = nested_cv_score(clf, X, y, n_outer=5, stratified=True)
    mlflow.log_metrics({
        "nested_cv_mean_auc": nested_cv["scores"].mean(),
        "nested_cv_std_auc":  nested_cv["scores"].std(),
        "nested_cv_mean_mcc": nested_cv["mccs"].mean(),
        "nested_cv_std_mcc":  nested_cv["mccs"].std(),
        "nested_cv_mean_threshold": nested_cv["optimal_thresholds"].mean(),
        "nested_cv_std_threshold":  nested_cv["optimal_thresholds"].std(),
    })

    # ─── Nested CV per fold 
    for i, (score, mcc, threshold, n_tree, params) in enumerate(zip(
        nested_cv["scores"],
        nested_cv["mccs"],
        nested_cv["optimal_thresholds"],
        nested_cv["n_trees"],
        nested_cv["best_params"]
    )):
        mlflow.log_metrics({
            f"fold_{i+1}_auc":       score,
            f"fold_{i+1}_mcc":       mcc,
            f"fold_{i+1}_threshold": threshold,
            f"fold_{i+1}_n_trees":   n_tree,
        })
        mlflow.log_dict(params, f"fold_{i+1}_params.json")

    with tempfile.TemporaryDirectory() as tmp:

        # ─── Thresholds & MCC across folds 
        threshold_df = pd.DataFrame({
            "fold":      range(1, len(nested_cv["optimal_thresholds"]) + 1),
            "threshold": nested_cv["optimal_thresholds"],
            "auc":       nested_cv["scores"],
            "mcc":       nested_cv["mccs"],
        })
        threshold_df.loc["mean"] = ["mean",
                                     nested_cv["optimal_thresholds"].mean(),
                                     nested_cv["scores"].mean(),
                                     nested_cv["mccs"].mean()]
        threshold_df.loc["std"]  = ["std",
                                     nested_cv["optimal_thresholds"].std(),
                                     nested_cv["scores"].std(),
                                     nested_cv["mccs"].std()]
        print(threshold_df.to_string(index=False))
        path = os.path.join(tmp, "thresholds_across_folds.csv")
        threshold_df.to_csv(path, index=False)
        mlflow.log_artifact(path)

        # ─── Classification Reports 
        train_report = classification_report(y_dev,  clf.predict(X_dev),  output_dict=True)
        test_report  = classification_report(y_test, clf.predict(X_test), output_dict=True)
        for split, report in [("train", train_report), ("test", test_report)]:
            path = os.path.join(tmp, f"{split}_report.json")
            json.dump(report, open(path, "w"), indent=2)
            mlflow.log_artifact(path)

        # ─── ROC Curve 
        fpr, tpr, _ = roc_curve(y_test, clf.predict_proba(X_test)[:, 1])
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, label=f"AUC = {clf.score(X_test, y_test):.4f}", linewidth=2)
        ax.plot([0, 1], [0, 1], "--", color="gray")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curve")
        ax.legend()
        fig.savefig(os.path.join(tmp, "roc_curve.png"))
        mlflow.log_artifact(os.path.join(tmp, "roc_curve.png"))
        plt.close(fig)

        # ─── Optuna Plots 
        fig1, fig2, fig3, fig4 = clf.plot_optuna_insights()
        for name, fig in [
            ("optimization_history", fig1),
            ("param_importances",    fig2),
            ("slice",                fig3),
            ("parallel_coordinate",  fig4),
        ]:
            path = os.path.join(tmp, f"{name}.html")
            fig.write_html(path)
            mlflow.log_artifact(path)

        # ─── Log Model ────────────────────────────────────────────────────────
        mlflow.xgboost.log_model(clf.best_model_, name="model")

/Users/vkvutov/Documents/my_codes/git_projects/xgb-cat-optuna/xgb-optuna/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


fold  threshold      auc      mcc
   1   0.355501 0.860741 0.541962
   2   0.357445 0.882593 0.573461
   3   0.380548 0.818519 0.462165
   4   0.355007 0.807075 0.383491
   5   0.365197 0.803962 0.421069
mean   0.362740 0.834578 0.476429
 std   0.009627 0.031437 0.071576


/Users/vkvutov/Documents/my_codes/git_projects/xgb-cat-optuna/xgb-optuna/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/vkvutov/Documents/my_codes/git_projects/xgb-cat-optuna/xgb-optuna/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/vkvutov/Documents/my_codes/git_projects/xgb-cat-optuna/xgb-optuna/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in 